In [1]:
import dukascopy_python
from dukascopy_python import instruments as ins

matches = [x for x in dir(ins) if "XAU" in x.upper() or "GOLD" in x.upper()]
print(matches)

['INSTRUMENT_FX_METALS_XAU_USD']


In [2]:
from datetime import datetime

INSTR_NAME = [x for x in matches if "USD" in x][0]
INSTR = getattr(ins, INSTR_NAME)
print("Instrument utilisé :", INSTR_NAME)

df = dukascopy_python.fetch(
    INSTR,
    dukascopy_python.INTERVAL_HOUR_1,
    dukascopy_python.OFFER_SIDE_BID,
    datetime(2025, 1, 1),
    datetime(2025, 2, 1),
)

print("Lignes :", df.shape[0])
print("Période :", df.index.min(), "→", df.index.max(), "| tz :", df.index.tz)
print(df.dtypes)
df.head()

Instrument utilisé : INSTRUMENT_FX_METALS_XAU_USD
Lignes : 504
Période : 2025-01-01 23:00:00+00:00 → 2025-01-31 21:00:00+00:00 | tz : UTC
open      float64
high      float64
low       float64
close     float64
volume    float64
dtype: object


,open,high,low,close,volume
timestamp,,,,,
2025-01-01 23:00:00+00:00,2625.098,2626.005,2621.475,2623.555,0.76854
2025-01-02 00:00:00+00:00,2623.655,2627.765,2622.448,2625.185,1.25358
2025-01-02 01:00:00+00:00,2625.185,2634.148,2623.645,2632.735,3.60279
2025-01-02 02:00:00+00:00,2632.735,2636.298,2632.148,2634.498,2.38464
2025-01-02 03:00:00+00:00,2634.404,2635.198,2632.065,2633.448,1.59480


In [3]:
import time
from datetime import datetime
import pandas as pd

START = datetime(2015, 1, 1)
END   = datetime(2026, 7, 1)   # borne fixe → snapshot reproductible

frames = []
for year in range(START.year, END.year + 1):
    cs = max(datetime(year, 1, 1), START)
    ce = min(datetime(year + 1, 1, 1), END)
    if cs >= ce:
        continue
    t0 = time.time()
    part = dukascopy_python.fetch(
        INSTR,
        dukascopy_python.INTERVAL_HOUR_1,
        dukascopy_python.OFFER_SIDE_BID,
        cs, ce,
    )
    frames.append(part)
    print(f"{year}: {len(part):>6} barres  ({cs.date()} → {ce.date()})  {time.time()-t0:.1f}s")

raw = pd.concat(frames)
print("\nTOTAL brut:", len(raw), "barres")

2015:   5991 barres  (2015-01-01 → 2016-01-01)  1.6s
2016:   5924 barres  (2016-01-01 → 2017-01-01)  1.1s
2017:   5910 barres  (2017-01-01 → 2018-01-01)  0.7s
2018:   5906 barres  (2018-01-01 → 2019-01-01)  2.2s
2019:   5910 barres  (2019-01-01 → 2020-01-01)  1.1s
2020:   5927 barres  (2020-01-01 → 2021-01-01)  1.4s
2021:   5907 barres  (2021-01-01 → 2022-01-01)  1.2s
2022:   5915 barres  (2022-01-01 → 2023-01-01)  1.1s
2023:   5890 barres  (2023-01-01 → 2024-01-01)  1.0s
2024:   5937 barres  (2024-01-01 → 2025-01-01)  0.5s
2025:   5912 barres  (2025-01-01 → 2026-01-01)  0.5s
2026:   2912 barres  (2026-01-01 → 2026-07-01)  0.7s

TOTAL brut: 68041 barres


In [4]:
import pandas as pd

df = raw.copy()
df = df[~df.index.duplicated(keep="first")]
df = df.sort_index()

print("=== RAPPORT QUALITÉ ===")
print("Barres          :", len(df))
print("Période         :", df.index.min(), "→", df.index.max())
print("Fuseau          :", df.index.tz)
print("Doublons retirés:", len(raw) - len(df))
print("Index croissant :", df.index.is_monotonic_increasing)
print("NaN total       :", int(df.isna().sum().sum()))

bad = (
    (df["high"] < df[["open", "close"]].max(axis=1)) |
    (df["low"]  > df[["open", "close"]].min(axis=1)) |
    (df["high"] < df["low"])
)
print("Violations OHLC :", int(bad.sum()))

gaps = df.index.to_series().diff()
big = gaps[gaps > pd.Timedelta(hours=1)]
wk = big.index.weekday
weekend_like = big[(wk == 6) | (wk == 0)]
print("\nTrous > 1h      :", len(big), "(normal : pause quotidienne + week-ends)")
print("  ~ week-end     :", len(weekend_like))
print("  en semaine     :", len(big) - len(weekend_like))
print("\n10 plus grands trous (reprise → durée) :")
print(big.sort_values(ascending=False).head(10))

print("\nPrix close min/max :", round(df["close"].min(), 2), "/", round(df["close"].max(), 2))

=== RAPPORT QUALITÉ ===
Barres          : 68041
Période         : 2015-01-01 23:00:00+00:00 → 2026-06-30 23:00:00+00:00
Fuseau          : UTC
Doublons retirés: 0
Index croissant : True
NaN total       : 0
Violations OHLC : 0

Trous > 1h      : 2853 (normal : pause quotidienne + week-ends)
  ~ week-end     : 1169
  en semaine     : 1684

10 plus grands trous (reprise → durée) :
timestamp
2015-12-27 23:00:00+00:00   3 days 05:00:00
2020-12-27 23:00:00+00:00   3 days 05:00:00
2017-12-25 23:00:00+00:00   3 days 02:00:00
2022-04-17 22:00:00+00:00   3 days 02:00:00
2019-04-21 22:00:00+00:00   3 days 02:00:00
2017-01-02 23:00:00+00:00   3 days 02:00:00
2018-04-01 22:00:00+00:00   3 days 02:00:00
2025-04-20 22:00:00+00:00   3 days 02:00:00
2022-12-26 23:00:00+00:00   3 days 02:00:00
2023-04-09 22:00:00+00:00   3 days 02:00:00
Name: timestamp, dtype: timedelta64[ns]

Prix close min/max : 1050.01 / 5562.3


In [5]:
import hashlib, json
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone
from importlib.metadata import version as _pkgver

# Racine du projet (le notebook est dans notebooks/)
PROJECT = Path.cwd()
if PROJECT.name == "notebooks":
    PROJECT = PROJECT.parent
SNAP_DIR = PROJECT / "data" / "snapshots"
SNAP_DIR.mkdir(parents=True, exist_ok=True)

VERSION = "v1"
base = f"XAUUSD_H1_BID_2015-01-01_2026-06-30_{VERSION}"
csv_path = SNAP_DIR / f"{base}.csv"
manifest_path = SNAP_DIR / f"{base}.manifest.json"

# Contrôles recalculés → manifeste auto-suffisant et honnête
bad = (
    (df["high"] < df[["open", "close"]].max(axis=1)) |
    (df["low"]  > df[["open", "close"]].min(axis=1)) |
    (df["high"] < df["low"])
)
gaps = df.index.to_series().diff()
big = gaps[gaps > pd.Timedelta(hours=1)]

df.to_csv(csv_path)  # index = timestamp UTC en ISO
sha = hashlib.sha256(csv_path.read_bytes()).hexdigest()

manifest = {
    "instrument": "XAUUSD — or spot",
    "instrument_constant": INSTR_NAME,
    "source": "Dukascopy Bank SA (via dukascopy-python)",
    "library_version": _pkgver("dukascopy-python"),
    "offer_side": "BID",
    "interval": "H1",
    "timezone": "UTC",
    "start": str(df.index.min()),
    "end": str(df.index.max()),
    "n_bars": int(len(df)),
    "columns": list(df.columns),
    "price_close_min": round(float(df["close"].min()), 3),
    "price_close_max": round(float(df["close"].max()), 3),
    "nan_count": int(df.isna().sum().sum()),
    "ohlc_violations": int(bad.sum()),
    "gaps_over_1h": int(len(big)),
    "largest_gap": str(big.max()),
    "downloaded_at_utc": datetime.now(timezone.utc).isoformat(),
    "csv_file": csv_path.name,
    "sha256": sha,
    "version": VERSION,
    "note": "IMMUABLE — ne jamais réécrire. Toute nouvelle donnée = nouvelle version.",
}
with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

# Vérification round-trip : le fichier gelé se relit sans perte
check = pd.read_csv(csv_path, index_col=0, parse_dates=True)
assert len(check) == len(df) and hashlib.sha256(csv_path.read_bytes()).hexdigest() == sha

print("Snapshot gelé et relu sans perte :", len(check), "barres")
print("Fichier :", csv_path.name)
print("sha256  :", sha)
print("\n" + json.dumps(manifest, indent=2, ensure_ascii=False))

Snapshot gelé et relu sans perte : 68041 barres
Fichier : XAUUSD_H1_BID_2015-01-01_2026-06-30_v1.csv
sha256  : 1c9f7d0349f78879c7ae116febc797a0b225e69c77acdd9f2faac06a8df13606

{
  "instrument": "XAUUSD — or spot",
  "instrument_constant": "INSTRUMENT_FX_METALS_XAU_USD",
  "source": "Dukascopy Bank SA (via dukascopy-python)",
  "library_version": "4.0.1",
  "offer_side": "BID",
  "interval": "H1",
  "timezone": "UTC",
  "start": "2015-01-01 23:00:00+00:00",
  "end": "2026-06-30 23:00:00+00:00",
  "n_bars": 68041,
  "columns": [
    "open",
    "high",
    "low",
    "close",
    "volume"
  ],
  "price_close_min": 1050.008,
  "price_close_max": 5562.295,
  "nan_count": 0,
  "ohlc_violations": 0,
  "gaps_over_1h": 2853,
  "largest_gap": "3 days 05:00:00",
  "downloaded_at_utc": "2026-07-05T13:32:39.352749+00:00",
  "csv_file": "XAUUSD_H1_BID_2015-01-01_2026-06-30_v1.csv",
  "sha256": "1c9f7d0349f78879c7ae116febc797a0b225e69c77acdd9f2faac06a8df13606",
  "version": "v1",
  "note": "IMMUABL

In [6]:
import json, hashlib
import pandas as pd
from pathlib import Path

PROJECT = Path.cwd()
if PROJECT.name == "notebooks":
    PROJECT = PROJECT.parent
SNAP_DIR = PROJECT / "data" / "snapshots"

base = "XAUUSD_H1_BID_2015-01-01_2026-06-30_v1"
csv_path = SNAP_DIR / f"{base}.csv"
manifest = json.loads((SNAP_DIR / f"{base}.manifest.json").read_text())

sha_now = hashlib.sha256(csv_path.read_bytes()).hexdigest()
assert sha_now == manifest["sha256"], "SHA256 différent → snapshot modifié, on s'arrête !"
print("Intégrité OK — sha256 vérifié contre le manifeste.")

data = pd.read_csv(csv_path, index_col=0, parse_dates=True)
data.index.name = "timestamp"
print("Barres :", len(data), "| période :", data.index.min(), "→", data.index.max())
data.head(3)

Intégrité OK — sha256 vérifié contre le manifeste.
Barres : 68041 | période : 2015-01-01 23:00:00+00:00 → 2026-06-30 23:00:00+00:00


,open,high,low,close,volume
timestamp,,,,,
2015-01-01 23:00:00+00:00,1183.949,1187.403,1183.230,1186.664,0.09
2015-01-02 00:00:00+00:00,1186.684,1188.282,1186.069,1187.412,0.28
2015-01-02 01:00:00+00:00,1187.398,1187.398,1184.341,1184.981,0.15


In [7]:
import numpy as np

N_BREAK = 20     # fenêtre de cassure (barres)
ATR_LEN = 14

df = data.copy()

# True Range + ATR (moyenne simple, suffisant pour prototyper)
prev_close = df["close"].shift(1)
tr = pd.concat([
    df["high"] - df["low"],
    (df["high"] - prev_close).abs(),
    (df["low"]  - prev_close).abs(),
], axis=1).max(axis=1)
df["atr"] = tr.rolling(ATR_LEN).mean()

# Cassure sur les N barres PRÉCÉDENTES — shift(1) = aucun lookahead
hh = df["high"].shift(1).rolling(N_BREAK).max()
ll = df["low"].shift(1).rolling(N_BREAK).min()
bull = df["close"] > hh
bear = df["close"] < ll

# Signaux « frais » = uniquement le passage de False à True
df["signal"] = 0
df.loc[bull & ~bull.shift(1, fill_value=False), "signal"] =  1
df.loc[bear & ~bear.shift(1, fill_value=False), "signal"] = -1

df = df.dropna(subset=["atr"])

n_long  = int((df["signal"] ==  1).sum())
n_short = int((df["signal"] == -1).sum())
print(f"Signaux BUY  : {n_long}")
print(f"Signaux SELL : {n_short}")
print(f"Total        : {n_long + n_short}  sur {len(df)} barres")
print(f"ATR médian   : {df['atr'].median():.2f} $")

Signaux BUY  : 2691
Signaux SELL : 2107
Total        : 4798  sur 68028 barres
ATR médian   : 3.82 $


In [8]:
import numpy as np
import pandas as pd

# --- Paramètres du harnais (tous ajustables) ---
SL_MULT  = 1.5      # stop  = 1.5 x ATR  → définit 1 R
TP_MULT  = 3.0      # target = 3.0 x ATR → cible 2 R
MAX_HOLD = 100      # sortie "temps" si ni SL ni TP après 100 barres (~4 jours)
COST_USD = 0.40     # coût round-turn conservateur (spread + commission), en $

o, h, l, c = (df[x].to_numpy() for x in ["open", "high", "low", "close"])
atr, sig   = df["atr"].to_numpy(), df["signal"].to_numpy()
idx, n     = df.index, len(df)

trades, i = [], 0
while i < n - 1:
    s = sig[i]
    if s != 0 and not np.isnan(atr[i]):
        d = 1 if s == 1 else -1
        entry = o[i + 1]                       # entrée à l'ouverture de la barre SUIVANTE
        R_usd = SL_MULT * atr[i]               # 1 R en dollars
        sl = entry - d * SL_MULT * atr[i]
        tp = entry + d * TP_MULT * atr[i]

        exit_price, reason, exit_j = None, None, None
        end = min(i + 1 + MAX_HOLD, n)
        for j in range(i + 1, end):
            hit_sl = (l[j] <= sl) if d == 1 else (h[j] >= sl)
            hit_tp = (h[j] >= tp) if d == 1 else (l[j] <= tp)
            if hit_sl and hit_tp:              # les deux dans la même barre → SL d'abord (conservateur)
                exit_price, reason, exit_j = sl, "SL_conservateur", j; break
            if hit_sl:
                exit_price, reason, exit_j = sl, "SL", j; break
            if hit_tp:
                exit_price, reason, exit_j = tp, "TP", j; break
        if exit_price is None:                 # sortie temps
            exit_j = end - 1
            exit_price, reason = c[exit_j], "TIME"

        R = (d * (exit_price - entry) - COST_USD) / R_usd
        trades.append({"entry_time": idx[i + 1], "dir": d, "reason": reason,
                       "bars_held": exit_j - i, "R": R})
        i = exit_j + 1                          # un seul trade à la fois : reprise après la sortie
    else:
        i += 1

tr = pd.DataFrame(trades).set_index("entry_time")
print("Trades exécutés :", len(tr), f"(sur {(sig != 0).sum()} signaux — reste filtré par 'un seul trade à la fois')")
print("\nSorties :"); print(tr["reason"].value_counts())
print(f"\nWin rate        : {(tr['R'] > 0).mean():.1%}")
print(f"Espérance/trade : {tr['R'].mean():+.3f} R   (moyenne BRUTE — pas encore un verdict)")
print(f"Somme totale    : {tr['R'].sum():+.1f} R")
print(f"Barres tenues (médiane) : {tr['bars_held'].median():.0f}")
tr.head()

Trades exécutés : 2662 (sur 4798 signaux — reste filtré par 'un seul trade à la fois')

Sorties :
reason
SL                 1735
TP                  905
SL_conservateur      16
TIME                  6
Name: count, dtype: int64

Win rate        : 34.2%
Espérance/trade : -0.052 R   (moyenne BRUTE — pas encore un verdict)
Somme totale    : -139.4 R
Barres tenues (médiane) : 6


,dir,reason,bars_held,R
entry_time,,,,
2015-01-05 06:00:00+00:00,1,SL,5,-1.057533
2015-01-05 15:00:00+00:00,1,TP,18,1.940290
2015-01-06 10:00:00+00:00,1,SL,2,-1.096389
2015-01-06 17:00:00+00:00,1,SL,13,-1.057475
2015-01-08 04:00:00+00:00,-1,SL,10,-1.076729


In [9]:
import numpy as np
import pandas as pd
from scipy import stats

R = tr["R"].to_numpy()
n = len(R)

# --- IC 95 % de l'espérance-R (bootstrap, aucune hypothèse de normalité) ---
rng = np.random.default_rng(42)
boot = np.array([rng.choice(R, size=n, replace=True).mean() for _ in range(10000)])
lo, hi = np.percentile(boot, [2.5, 97.5])

print("=== STRATÉGIE : cassure 20 barres ===")
print(f"Trades          : {n}")
print(f"Espérance-R     : {R.mean():+.4f} R")
print(f"IC 95 %         : [{lo:+.4f} , {hi:+.4f}] R")
print(f"Exclut zéro ?   : {'OUI' if (lo > 0 or hi < 0) else 'NON — indistinguable de zéro'}")

# --- TÉMOIN ALÉATOIRE : mêmes SL/TP/coûts, mais entrées tirées au sort ---
# On rejoue le simulateur avec des signaux aléatoires, à fréquence égale.
o, h, l, c = (df[x].to_numpy() for x in ["open","high","low","close"])
atr = df["atr"].to_numpy()
N = len(df)
p_signal = (df["signal"] != 0).mean()   # même densité de signaux que la vraie stratégie

def run_random(seed):
    rng = np.random.default_rng(seed)
    rand_sig = np.where(rng.random(N) < p_signal, rng.choice([-1, 1], size=N), 0)
    out, i = [], 0
    while i < N - 1:
        s = rand_sig[i]
        if s != 0 and not np.isnan(atr[i]):
            d = int(s); entry = o[i+1]; R_usd = SL_MULT*atr[i]
            sl = entry - d*SL_MULT*atr[i]; tp = entry + d*TP_MULT*atr[i]
            ep, ej = None, None
            end = min(i+1+MAX_HOLD, N)
            for j in range(i+1, end):
                hit_sl = (l[j]<=sl) if d==1 else (h[j]>=sl)
                hit_tp = (h[j]>=tp) if d==1 else (l[j]<=tp)
                if hit_sl: ep, ej = sl, j; break
                if hit_tp: ep, ej = tp, j; break
            if ep is None: ej = end-1; ep = c[ej]
            out.append((d*(ep-entry) - COST_USD)/R_usd)
            i = ej + 1
        else:
            i += 1
    return np.mean(out)

rand_means = np.array([run_random(s) for s in range(200)])
r_lo, r_hi = np.percentile(rand_means, [2.5, 97.5])
pct = (rand_means < R.mean()).mean()

print("\n=== TÉMOIN ALÉATOIRE (200 tirages) ===")
print(f"Espérance-R moyenne du hasard : {rand_means.mean():+.4f} R")
print(f"Plage 95 % du hasard          : [{r_lo:+.4f} , {r_hi:+.4f}] R")
print(f"\nLa stratégie ({R.mean():+.4f} R) bat le hasard dans {1-pct:.0%} des cas.")

print("\n=== VERDICT ===")
if lo > 0 and R.mean() > r_hi:
    print("EDGE PROBABLE : positif, significatif, et au-dessus du hasard.")
elif hi < 0:
    print("PERDANT SIGNIFICATIF : espérance négative, IC exclut zéro.")
else:
    print("PAS D'EDGE : indistinguable de zéro et/ou du hasard. Hypothèse #1 rejetée — on itère.")

=== STRATÉGIE : cassure 20 barres ===
Trades          : 2662
Espérance-R     : -0.0524 R
IC 95 %         : [-0.1062 , +0.0026] R
Exclut zéro ?   : NON — indistinguable de zéro

=== TÉMOIN ALÉATOIRE (200 tirages) ===
Espérance-R moyenne du hasard : -0.0785 R
Plage 95 % du hasard          : [-0.1262 , -0.0271] R

La stratégie (-0.0524 R) bat le hasard dans 14% des cas.

=== VERDICT ===
PAS D'EDGE : indistinguable de zéro et/ou du hasard. Hypothèse #1 rejetée — on itère.


In [10]:
import numpy as np
import pandas as pd

SL_MULT, TP_MULT, MAX_HOLD, COST_USD = 1.5, 3.0, 100, 0.40

def run_backtest(signal, df):
    """Un trade à la fois, entrée barre suivante, SL/TP en ATR. Renvoie les R par trade."""
    o, h, l, c = (df[x].to_numpy() for x in ["open","high","low","close"])
    atr = df["atr"].to_numpy(); sig = np.asarray(signal); n = len(df)
    R_list, i = [], 0
    while i < n - 1:
        s = sig[i]
        if s != 0 and not np.isnan(atr[i]):
            d = 1 if s == 1 else -1
            entry = o[i+1]; R_usd = SL_MULT*atr[i]
            sl = entry - d*SL_MULT*atr[i]; tp = entry + d*TP_MULT*atr[i]
            ep, ej = None, None; end = min(i+1+MAX_HOLD, n)
            for j in range(i+1, end):
                hit_sl = (l[j] <= sl) if d==1 else (h[j] >= sl)
                hit_tp = (h[j] >= tp) if d==1 else (l[j] <= tp)
                if hit_sl: ep, ej = sl, j; break
                if hit_tp: ep, ej = tp, j; break
            if ep is None: ej = end-1; ep = c[ej]
            R_list.append((d*(ep-entry) - COST_USD)/R_usd)
            i = ej + 1
        else:
            i += 1
    return np.array(R_list)

def evaluate(signal, df, label, n_boot=10000, n_random=100, seed=0):
    signal = np.asarray(signal)
    R = run_backtest(signal, df); n = len(R)
    rng = np.random.default_rng(seed)
    boot = np.array([rng.choice(R, n, replace=True).mean() for _ in range(n_boot)])
    lo, hi = np.percentile(boot, [2.5, 97.5])
    # Témoin APPARIÉ : même densité de signaux ET même biais long/court
    mask = signal != 0; density = mask.mean()
    p_long = (signal == 1).sum() / max(mask.sum(), 1); N = len(df)
    def one_random(s):
        r = np.random.default_rng(s)
        rand_sig = np.where(r.random(N) < density,
                            np.where(r.random(N) < p_long, 1, -1), 0)
        return run_backtest(rand_sig, df).mean()
    rand = np.array([one_random(1000+s) for s in range(n_random)])
    return {"label": label, "trades": n, "long_frac": round(float(p_long),2),
            "esperance_R": round(float(R.mean()),4),
            "IC": [round(float(lo),4), round(float(hi),4)],
            "exclut_0": bool(lo > 0 or hi < 0),
            "hasard": round(float(rand.mean()),4),
            "bat_hasard": round(float((R.mean() > rand).mean()),2)}

# Fidélité : doit redonner ~2662 trades et -0.0524 R
print(evaluate(df["signal"].values, df, "Baseline (cassure seule)"))

{'label': 'Baseline (cassure seule)', 'trades': 2662, 'long_frac': 0.56, 'esperance_R': -0.0524, 'IC': [-0.107, 0.0019], 'exclut_0': False, 'hasard': -0.0721, 'bat_hasard': 0.76}


In [11]:
# Filtre A : EMA200 sur H1 (tendance classique)
df["ema200"] = df["close"].ewm(span=200, adjust=False).mean()
df["htf_ema"] = np.where(df["close"] > df["ema200"], 1, -1)

# Filtre B : structure HTF (biais Daily via BOS sur pivots, SANS lookahead)
daily = (df.resample("1D")
           .agg({"open":"first","high":"max","low":"min","close":"last"}).dropna())
W = 3
Hd, Ld, Cd = daily["high"].values, daily["low"].values, daily["close"].values
nD = len(daily)
is_ph = np.zeros(nD, bool); is_pl = np.zeros(nD, bool)
for k in range(W, nD - W):
    if Hd[k] == Hd[k-W:k+W+1].max(): is_ph[k] = True
    if Ld[k] == Ld[k-W:k+W+1].min(): is_pl[k] = True

bias = np.zeros(nD, int); last_sh, last_sl, cur = np.nan, np.nan, 0
for t in range(nD):
    k = t - W                                   # pivot confirmé W barres plus tard
    if k >= 0:
        if is_ph[k]: last_sh = Hd[k]
        if is_pl[k]: last_sl = Ld[k]
    if not np.isnan(last_sh) and Cd[t] > last_sh: cur = 1      # BOS haussier
    elif not np.isnan(last_sl) and Cd[t] < last_sl: cur = -1   # BOS baissier
    bias[t] = cur
daily["bias"] = bias

# Le biais du jour D n'est utilisable qu'à partir de D+1 (lookahead_off)
lookup = pd.DataFrame({"known_from": daily.index + pd.Timedelta(days=1),
                       "bias": daily["bias"].values})
m = pd.merge_asof(df.reset_index()[["timestamp"]], lookup,
                  left_on="timestamp", right_on="known_from", direction="backward")
df["htf_struct"] = m["bias"].fillna(0).astype(int).values

print("A (EMA200)    — bull:", int((df.htf_ema==1).sum()), "| bear:", int((df.htf_ema==-1).sum()))
print("B (structure) — bull:", int((df.htf_struct==1).sum()),
      "| bear:", int((df.htf_struct==-1).sum()), "| neutre:", int((df.htf_struct==0).sum()))
nz = df.htf_struct != 0
print(f"Accord A/B (hors neutre) : {((df.htf_ema==df.htf_struct)&nz).sum()/nz.sum():.0%}")

A (EMA200)    — bull: 37791 | bear: 30237
B (structure) — bull: 36553 | bear: 30878 | neutre: 597
Accord A/B (hors neutre) : 67%


In [12]:
import pandas as pd

base_sig = df["signal"].values
ema  = df["htf_ema"].values
strc = df["htf_struct"].values

# On ne garde le signal que s'il va DANS LE SENS du filtre
sig_A = np.where(base_sig == ema,  base_sig, 0)   # EMA200
sig_B = np.where(base_sig == strc, base_sig, 0)    # structure Daily

results = [
    evaluate(base_sig, df, "0. Baseline (cassure seule)"),
    evaluate(sig_A,    df, "A. + filtre EMA200"),
    evaluate(sig_B,    df, "B. + filtre structure Daily"),
]

res = pd.DataFrame(results)[
    ["label","trades","long_frac","esperance_R","IC","exclut_0","hasard","bat_hasard"]
]
pd.set_option("display.width", 200, "display.max_colwidth", 40)
print(res.to_string(index=False))

print("\n--- Lecture ---")
for r in results:
    edge = r["exclut_0"] and r["esperance_R"] > 0 and r["bat_hasard"] >= 0.95
    verdict = "EDGE PROBABLE" if edge else ("indistinguable" )
    print(f"{r['label']:32s} {r['esperance_R']:+.4f} R  IC={r['IC']}  bat_hasard={r['bat_hasard']:.0%}  → {verdict}")

                      label  trades  long_frac  esperance_R                IC  exclut_0  hasard  bat_hasard
0. Baseline (cassure seule)    2662       0.56      -0.0524  [-0.107, 0.0019]     False -0.0721        0.76
         A. + filtre EMA200    2185       0.59      -0.0461 [-0.1058, 0.0155]     False -0.0680        0.80
B. + filtre structure Daily    1508       0.60      -0.0305 [-0.1036, 0.0411]     False -0.0687        0.92

--- Lecture ---
0. Baseline (cassure seule)      -0.0524 R  IC=[-0.107, 0.0019]  bat_hasard=76%  → indistinguable
A. + filtre EMA200               -0.0461 R  IC=[-0.1058, 0.0155]  bat_hasard=80%  → indistinguable
B. + filtre structure Daily      -0.0305 R  IC=[-0.1036, 0.0411]  bat_hasard=92%  → indistinguable


In [13]:
# --- Couche 2 : zone discount/premium (équilibre 50% du dernier swing HTF) ---
daily = (df.resample("1D")
           .agg({"open":"first","high":"max","low":"min","close":"last"}).dropna())
W = 3
Hd, Ld = daily["high"].values, daily["low"].values
nD = len(daily)
is_ph = np.zeros(nD, bool); is_pl = np.zeros(nD, bool)
for k in range(W, nD - W):
    if Hd[k] == Hd[k-W:k+W+1].max(): is_ph[k] = True
    if Ld[k] == Ld[k-W:k+W+1].min(): is_pl[k] = True

sh_arr = np.full(nD, np.nan); sl_arr = np.full(nD, np.nan)
last_sh, last_sl = np.nan, np.nan
for t in range(nD):
    k = t - W                                   # pivot confirmé W barres plus tard (lookahead_off)
    if k >= 0:
        if is_ph[k]: last_sh = Hd[k]
        if is_pl[k]: last_sl = Ld[k]
    sh_arr[t], sl_arr[t] = last_sh, last_sl

lookup = pd.DataFrame({"known_from": daily.index + pd.Timedelta(days=1),
                       "sh": sh_arr, "sl": sl_arr})
m = pd.merge_asof(df.reset_index()[["timestamp"]], lookup,
                  left_on="timestamp", right_on="known_from", direction="backward")
df["sh"], df["sl"] = m["sh"].values, m["sl"].values

eq = (df["sh"] + df["sl"]) / 2
discount = (df["close"] < eq).values     # moitié basse du swing
premium  = (df["close"] > eq).values     # moitié haute du swing

# BUY seulement en discount, SELL seulement en premium
in_zone = np.where(df["htf_struct"].values == 1, discount,
           np.where(df["htf_struct"].values == -1, premium, False)).astype(bool)
sig_B_zone = np.where((sig_B != 0) & in_zone, sig_B, 0)

print(f"Barres avec swing HTF défini : {df[['sh','sl']].notna().all(axis=1).mean():.0%}")
print(f"Signaux B        : {(sig_B != 0).sum()}")
print(f"Signaux B + zone : {(sig_B_zone != 0).sum()}")

results2 = [
    evaluate(sig_B,      df, "B. structure Daily (référence)"),
    evaluate(sig_B_zone, df, "C. B + discount (Fib 50%)"),
]
res2 = pd.DataFrame(results2)[
    ["label","trades","long_frac","esperance_R","IC","exclut_0","hasard","bat_hasard"]
]
print()
print(res2.to_string(index=False))

print("\n--- Lecture ---")
for r in results2:
    edge = r["exclut_0"] and r["esperance_R"] > 0 and r["bat_hasard"] >= 0.95
    print(f"{r['label']:34s} {r['esperance_R']:+.4f} R  IC={r['IC']}  "
          f"bat_hasard={r['bat_hasard']:.0%}  → {'EDGE PROBABLE' if edge else 'indistinguable'}")

Barres avec swing HTF défini : 99%
Signaux B        : 2550
Signaux B + zone : 330

                         label  trades  long_frac  esperance_R                 IC  exclut_0  hasard  bat_hasard
B. structure Daily (référence)    1508       0.60      -0.0305  [-0.1036, 0.0411]     False -0.0687        0.92
     C. B + discount (Fib 50%)     241       0.55      -0.2211 [-0.3928, -0.0452]      True -0.0702        0.01

--- Lecture ---
B. structure Daily (référence)     -0.0305 R  IC=[-0.1036, 0.0411]  bat_hasard=92%  → indistinguable
C. B + discount (Fib 50%)          -0.2211 R  IC=[-0.3928, -0.0452]  bat_hasard=1%  → indistinguable


In [14]:
o = df["open"].values; h = df["high"].values; l = df["low"].values; c = df["close"].values
eq   = ((df["sh"] + df["sl"]) / 2).values      # équilibre 50% du dernier swing HTF
bias = df["htf_struct"].values                 # biais Daily (B)

body       = np.abs(c - o)
lower_wick = np.minimum(o, c) - l
upper_wick = h - np.maximum(o, c)

in_discount = c < eq       # clôture dans la moitié basse du swing
in_premium  = c > eq       # clôture dans la moitié haute

# Référence : cassure + structure (le meilleur résultat précédent)
base_sig = df["signal"].values
sig_B = np.where(base_sig == bias, base_sig, 0)

# D — Rejet simple : tendance + repli en zone + bougie dans le sens de la tendance
bull_simple = (bias == 1)  & in_discount & (c > o)
bear_simple = (bias == -1) & in_premium  & (c < o)
sig_simple  = np.where(bull_simple, 1, np.where(bear_simple, -1, 0))

# E — Rejet de mèche : sous-ensemble strict (mèche de rejet >= corps)
bull_wick = bull_simple & (lower_wick >= body)
bear_wick = bear_simple & (upper_wick >= body)
sig_wick  = np.where(bull_wick, 1, np.where(bear_wick, -1, 0))

print(f"Signaux B (référence, cassure) : {(sig_B != 0).sum()}")
print(f"Signaux

SyntaxError: unterminated string literal (detected at line 27) (3923446961.py, line 27)

In [15]:
o = df["open"].values; h = df["high"].values; l = df["low"].values; c = df["close"].values
eq   = ((df["sh"] + df["sl"]) / 2).values      # équilibre 50% du dernier swing HTF
bias = df["htf_struct"].values                 # biais Daily (B)

body       = np.abs(c - o)
lower_wick = np.minimum(o, c) - l
upper_wick = h - np.maximum(o, c)

in_discount = c < eq       # clôture dans la moitié basse du swing
in_premium  = c > eq       # clôture dans la moitié haute

base_sig = df["signal"].values
sig_B = np.where(base_sig == bias, base_sig, 0)

bull_simple = (bias == 1)  & in_discount & (c > o)
bear_simple = (bias == -1) & in_premium  & (c < o)
sig_simple  = np.where(bull_simple, 1, np.where(bear_simple, -1, 0))

bull_wick = bull_simple & (lower_wick >= body)
bear_wick = bear_simple & (upper_wick >= body)
sig_wick  = np.where(bull_wick, 1, np.where(bear_wick, -1, 0))

n_B = int((sig_B != 0).sum())
n_D = int((sig_simple != 0).sum())
n_E = int((sig_wick != 0).sum())
print("Signaux B (reference, cassure) :", n_B)
print("Signaux D (rejet simple)       :", n_D)
print("Signaux E (rejet meche)        :", n_E)

results3 = [
    evaluate(sig_B,      df, "B. cassure + structure (ref.)"),
    evaluate(sig_simple, df, "D. rejet simple en discount"),
    evaluate(sig_wick,   df, "E. rejet de meche (sous-ensemble)"),
]
res3 = pd.DataFrame(results3)[
    ["label","trades","long_frac","esperance_R","IC","exclut_0","hasard","bat_hasard"]
]
print()
print(res3.to_string(index=False))

print("\n--- Lecture (regle fixee d'avance) ---")
for r in results3:
    edge = r["exclut_0"] and r["esperance_R"] > 0 and r["bat_hasard"] >= 0.95
    lisible = r["trades"] >= 300
    tag = "EDGE PROBABLE" if edge else ("indistinguable" if lisible else "TROP PEU DE TRADES")
    print(f"{r['label']:34s} {r['esperance_R']:+.4f} R  IC={r['IC']}  "
          f"n={r['trades']}  bat_hasard={r['bat_hasard']:.0%}  -> {tag}")

Signaux B (reference, cassure) : 2550
Signaux D (rejet simple)       : 8859
Signaux E (rejet meche)        : 3227

                            label  trades  long_frac  esperance_R                 IC  exclut_0  hasard  bat_hasard
    B. cassure + structure (ref.)    1508        0.6      -0.0305  [-0.1036, 0.0411]     False -0.0687        0.92
      D. rejet simple en discount    1424        0.5      -0.1376 [-0.2096, -0.0647]      True -0.0771        0.01
E. rejet de meche (sous-ensemble)    1103        0.5      -0.1195 [-0.2006, -0.0371]      True -0.0742        0.07

--- Lecture (regle fixee d'avance) ---
B. cassure + structure (ref.)      -0.0305 R  IC=[-0.1036, 0.0411]  n=1508  bat_hasard=92%  -> indistinguable
D. rejet simple en discount        -0.1376 R  IC=[-0.2096, -0.0647]  n=1424  bat_hasard=1%  -> indistinguable
E. rejet de meche (sous-ensemble)  -0.1195 R  IC=[-0.2006, -0.0371]  n=1103  bat_hasard=7%  -> indistinguable


In [16]:
import numpy as np
import pandas as pd
from scipy import stats

work = data.copy()                          # snapshot vérifié (cellule 6)
work["ret"] = work["close"].pct_change()    # rendement H1
work["hour"] = work.index.hour              # heure UTC

rows = []
for hr, g in work.groupby("hour"):
    r = g["ret"].dropna().values
    if len(r) < 30:
        continue
    t, p = stats.ttest_1samp(r, 0.0)        # ce rendement horaire diffère-t-il de 0 ?
    rows.append({
        "heure_UTC": hr,
        "n": len(r),
        "ret_moyen_bp": round(r.mean() * 1e4, 2),   # en points de base
        "ret_median_bp": round(np.median(r) * 1e4, 2),
        "vol_bp": round(r.std() * 1e4, 1),
        "t_stat": round(float(t), 2),
        "p_value": round(float(p), 4),
    })

tab = pd.DataFrame(rows)
pd.set_option("display.width", 200)
print(tab.to_string(index=False))

signif = tab[tab["p_value"] < 0.05].sort_values("t_stat")
print(f"\nHeures au rendement significativement != 0 (p<0.05) : {len(signif)} / {len(tab)}")
if len(signif):
    print(signif[["heure_UTC","n","ret_moyen_bp","t_stat","p_value"]].to_string(index=False))
else:
    print("Aucune heure ne se détache du bruit.")

 heure_UTC    n  ret_moyen_bp  ret_median_bp  vol_bp  t_stat  p_value
         0 2965         -0.10          -0.43    16.9   -0.32   0.7527
         1 2965          0.85           0.68    23.7    1.96   0.0502
         2 2965          0.48           0.43    17.0    1.55   0.1218
         3 2965         -0.21          -0.45    13.9   -0.84   0.4028
         4 2965         -0.07          -0.37    10.7   -0.33   0.7379
         5 2965          0.11           1.09    17.5    0.34   0.7328
         6 2965          0.49          -0.28    18.5    1.44   0.1501
         7 2965         -0.01          -0.03    18.7   -0.04   0.9667
         8 2964          0.10           0.61    18.1    0.29   0.7732
         9 2963         -0.06           0.09    16.0   -0.22   0.8275
        10 2965         -0.12           0.17    16.7   -0.40   0.6882
        11 2965          0.10           0.33    18.3    0.29   0.7695
        12 2964          0.31           0.24    28.6    0.58   0.5593
        13 2964     

In [17]:
import numpy as np
import pandas as pd

w = data.copy()
w["hour"] = w.index.hour
w["date"] = w.index.normalize()

ENTRY_H, EXIT_H = 21, 23
COST_NORMAL, COST_ILLIQ = 0.40, 0.80

# Table pivot : une ligne par jour, une colonne par heure de close (rapide, vectorisé)
piv = w.pivot_table(index="date", columns="hour", values="close", aggfunc="last")
day_scale = w.groupby("date")["close"].std()   # échelle du jour pour normaliser en R

def sess_R(entry_h, exit_h, cost):
    if entry_h not in piv.columns or exit_h not in piv.columns:
        return np.array([])
    d = pd.DataFrame({"e": piv[entry_h], "x": piv[exit_h], "s": day_scale}).dropna()
    d = d[d["s"] > 0]
    return (((d["x"] - d["e"]) - cost) / d["s"]).values

def judge(R, label, seed=0, n_random=300):
    n = len(R)
    if n == 0:
        print(label, ": aucun trade"); return
    rng = np.random.default_rng(seed)
    boot = np.array([rng.choice(R, n, replace=True).mean() for _ in range(5000)])
    lo, hi = np.percentile(boot, [2.5, 97.5])
    hours = list(piv.columns)
    rnd = []
    for s in range(n_random):
        eh = np.random.default_rng(1000+s).choice(hours)
        rr = sess_R(eh, (eh + 2) % 24, R_cost_random)
        if len(rr): rnd.append(rr.mean())
    rnd = np.array(rnd)
    beat = float((R.mean() > rnd).mean()) if len(rnd) else float("nan")
    print(f"{label:26s} n={n}  esp={R.mean():+.4f} R  "
          f"IC=[{lo:+.4f},{hi:+.4f}]  exclut0={'OUI' if (lo>0 or hi<0) else 'non'}  "
          f"bat_hasard={beat:.0%}")

print("Strategie : LONG close 21h UTC -> close 23h UTC\n")
R_cost_random = COST_NORMAL
judge(sess_R(ENTRY_H, EXIT_H, COST_NORMAL), "Cout normal (0.40$)")
R_cost_random = COST_ILLIQ
judge(sess_R(ENTRY_H, EXIT_H, COST_ILLIQ), "Cout heure creuse (0.80$)")

Rn = sess_R(ENTRY_H, EXIT_H, COST_NORMAL)
print(f"\nRendement brut moyen/trade : {Rn.mean():+.4f} R  (avant majoration cout)")

Strategie : LONG close 21h UTC -> close 23h UTC

Cout normal (0.40$)        n=834  esp=+0.0734 R  IC=[+0.0438,+0.1031]  exclut0=OUI  bat_hasard=96%
Cout heure creuse (0.80$)  n=834  esp=-0.0358 R  IC=[-0.0658,-0.0055]  exclut0=OUI  bat_hasard=96%

Rendement brut moyen/trade : +0.0734 R  (avant majoration cout)
